# Лабораторна робота 5 — Логістична регресія для аналізу тональності

**Набори даних:** `amazon_baby_subset.csv`, `important_words.json`  
**Обмеження:** scikit-learn-класифікатори **не дозволені** для базових завдань.

## Налаштування

In [1]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib --quiet


In [2]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

%matplotlib inline


## Теоретичне підґрунтя

Сигмоїдна функція:
```
P(y = +1 | x, w) = 1 / (1 + exp(−wᵀ h(x)))
```
Похідна логарифму правдоподібності відносно wⱼ:
```
d(ll)/dwⱼ = dot( hⱼ,  1[y=+1] − P(y=+1|x,w) )
```
де 1[y=+1] = 1, якщо мітка +1, інакше 0.

---
## Завдання 1 — Підготовка ознак

1. Завантажте `amazon_baby_subset.csv`. Видаліть рядки з відсутніми відгуками. Вилучіть відгуки з `rating == 3`. Створіть стовпець `sentiment`: **+1** якщо `rating >= 4`, інакше **−1**.
2. Завантажте `important_words.json` (193 слова). Для кожного слова додайте стовпець до DataFrame із підрахунком його входжень у очищений текст відгуку.
3. Повідомте, скільки відгуків залишилось та який баланс класів.

In [5]:
products = pd.read_csv('/kaggle/input/datasets/dariatopchii/mtad-lab5-ds/amazon_baby_subset.csv')

products = products.dropna(subset=['review'])

products = products[products['rating'] != 3].copy()

products['sentiment'] = products['rating'].apply(lambda rating: +1 if rating >= 4 else -1)

print(f'Усього відгуків : {len(products)}')
print(f'Позитивні (+1)  : {(products["sentiment"] == 1).sum()}')
print(f'Негативні (-1)  : {(products["sentiment"] == -1).sum()}')

Усього відгуків : 17311
Позитивні (+1)  : 17310
Негативні (-1)  : 1


In [8]:
# Завантаження важливих слів
with open('/kaggle/input/datasets/dariatopchii/mtad-lab5-ds/important_words.json') as f:
    important_words = json.load(f)
print(f'Розмір словника: {len(important_words)} слів')


Розмір словника: 193 слів


In [21]:
import string

products['review_clean'] = (
    products['review']
    .fillna('')
    .str.replace(f'[{string.punctuation}]', '', regex=True)
    .str.lower()
)

review_words = products['review_clean'].str.split()

word_counts = pd.DataFrame(
    {
        word: review_words.apply(lambda words: words.count(word))
        for word in important_words
    },
    index=products.index
)

products = products.drop(columns=[word for word in important_words if word in products.columns], errors='ignore')

products = pd.concat([products, word_counts], axis=1)

print('Приклад підрахунку слів:')
products[important_words[:5]].head(3)

Приклад підрахунку слів:


,baby,one,great,love,use
0,0,0,1,0,0
1,0,0,0,0,0
2,1,0,0,0,0


---
## Завдання 2 — Побудова матриці ознак

Реалізуйте `get_feature_matrix(df, word_list)`, яка:
1. Створює масив NumPy зі стовпцем одиниць (вільний член), за яким іде по одному стовпцю для кожного слова зі списку.
2. Повертає `(feature_matrix, sentiment_array)`, де `sentiment_array` містить +1 або −1.

Перевірка: `feature_matrix` має форму `(N, 194)`.

In [10]:
def get_feature_matrix(df, word_list):
    """
    Будує матрицю ознак та вектор міток тональності.

    Повертає
    -------
    feature_matrix  : np.ndarray, shape (n, len(word_list)+1)
    sentiment_array : np.ndarray, shape (n,)  значення {+1, -1}
    """
    intercept = np.ones((len(df), 1))

    word_features = df[word_list].to_numpy(dtype=float)

    feature_matrix = np.hstack((intercept, word_features))

    sentiment_array = df['sentiment'].to_numpy()
    return feature_matrix, sentiment_array

In [11]:
feature_matrix, sentiment = get_feature_matrix(products, important_words)
print(f'Розмір feature_matrix: {feature_matrix.shape}')  # очікується (N, 194)


Розмір feature_matrix: (17311, 194)


---
## Завдання 3 — Сигмоїдна функція та передбачення

Реалізуйте `predict_probability(feature_matrix, coefficients)`, що обчислює сигмоїдну функцію для кожного рядка. Результат — масив NumPy зі значеннями у (0, 1).

In [12]:
def predict_probability(feature_matrix, coefficients):
    """
    Обчислює P(y = +1 | x, w) для кожного рядка.

    Повертає
    -------
    probabilities : np.ndarray, shape (n,), значення у (0, 1)
    """
    score = np.dot(feature_matrix, coefficients)

    probabilities = 1 / (1 + np.exp(-score))

    return probabilities

### Перевірка — при нульових вхідних даних кожне передбачення має дорівнювати 0.5

In [13]:
zero_coeffs = np.zeros(feature_matrix.shape[1])
test_probs  = predict_probability(feature_matrix, zero_coeffs)
print(f'Усі передбачення 0.5: {np.allclose(test_probs, 0.5)}')


Усі передбачення 0.5: True


---
## Завдання 4 — Градієнтний підйом

Реалізуйте `logistic_regression(feature_matrix, sentiment, initial_coefficients, step_size, max_iter)`. На кожній ітерації:
1. Обчислюйте передбачення за допомогою `predict_probability()`.
2. Обчислюйте `errors = 1[y=+1] − predictions`.
3. Для кожного коефіцієнта j: `derivative = dot(feature_j, errors)`, потім `coeff[j] += step_size · derivative`.

Запустіть з: `initial_coefficients = np.zeros(194)`, `step_size = 1e-7`, `max_iter = 301`. Виводьте логарифм правдоподібності кожні 50 ітерацій — він має зростати монотонно.

In [14]:
def compute_log_likelihood(feature_matrix, sentiment, coefficients):
    indicator = (sentiment == +1).astype(float)
    scores    = np.dot(feature_matrix, coefficients)
    ll        = np.sum(
        indicator * scores - np.log(1.0 + np.exp(scores))
    )
    return ll


In [15]:
def logistic_regression(feature_matrix, sentiment,
                        initial_coefficients, step_size, max_iter):
    """
    Навчає ваги логістичної регресії методом градієнтного підйому.

    Повертає
    -------
    coefficients : np.ndarray, shape (n_features,)
    """
    coefficients = np.array(initial_coefficients, dtype=float)
    indicator    = (sentiment == +1).astype(float)   

    for itr in range(max_iter):
        predictions = predict_probability(feature_matrix, coefficients)

        errors = indicator - predictions

        for j in range(len(coefficients)):
            derivative = np.dot(feature_matrix[:, j], errors)
            coefficients[j] = coefficients[j] + step_size * derivative

        if itr % 50 == 0:
            ll = compute_log_likelihood(feature_matrix, sentiment, coefficients)
            print(f'Ітерація {itr:4d}  |  логарифм правдоподібності: {ll:.4f}')

    return coefficients

### Запуск моделі

In [16]:
coefficients = logistic_regression(
    feature_matrix, sentiment,
    initial_coefficients=np.zeros(194),
    step_size=1e-7,
    max_iter=301
)


Ітерація    0  |  логарифм правдоподібності: -11975.0356
Ітерація   50  |  логарифм правдоподібності: -10884.4567
Ітерація  100  |  логарифм правдоподібності: -9978.6843
Ітерація  150  |  логарифм правдоподібності: -9216.2813
Ітерація  200  |  логарифм правдоподібності: -8566.0665
Ітерація  250  |  логарифм правдоподібності: -8004.8996
Ітерація  300  |  логарифм правдоподібності: -7515.4908


### Точність класифікації та базовий рівень

In [17]:
# Передбачте мітки класів (+1 якщо score > 0, інакше -1)
scores          = np.dot(feature_matrix, coefficients)
predictions = np.where(scores > 0, +1, -1)

model_accuracy  = np.mean(predictions == sentiment)
print(f'Точність моделі: {model_accuracy:.4f}')

# Базовий рівень мажоритарного класу
majority_class = +1 if (sentiment == +1).sum() >= (sentiment == -1).sum() else -1
baseline_acc = np.mean(majority_class == sentiment)
print(f'Базовий рівень: {baseline_acc:.4f} (завжди передбачає {majority_class})')


Точність моделі: 0.9999
Базовий рівень: 0.9999 (завжди передбачає 1)


---
## ✨ Бонус — Інтерпретація моделі

Зіставте кожне слово з його навченим коефіцієнтом. Виведіть 10 слів з найбільшими коефіцієнтами та 10 слів з найменшими.

Для одного слова з кожного списку знайдіть відгук у наборі даних, що його містить, і вкажіть передбачувану ймовірність моделі.

In [18]:
coef_table = pd.DataFrame({
    'word': important_words,
    'coefficient': coefficients[1:]
})

positive_words = coef_table.sort_values('coefficient', ascending=False).head(10)
negative_words = coef_table.sort_values('coefficient', ascending=True).head(10)

print('10 найбільш позитивних слів:')
display(positive_words)

print('10 найбільш негативних слів:')
display(negative_words)

10 найбільш позитивних слів:


,word,coefficient
0,baby,0.080332
1,one,0.077519
2,great,0.075589
4,use,0.058326
5,would,0.056601
3,love,0.056481
7,easy,0.051770
8,little,0.051176
6,like,0.050968
10,old,0.046130


10 найбільш негативних слів:


,word,coefficient
189,won,0.000098
168,returned,0.000940
113,return,0.000962
112,waste,0.001154
171,broke,0.001319
105,disappointed,0.001671
182,unit,0.001878
166,company,0.002211
180,working,0.002712
133,cheap,0.002930


In [19]:
# Бонус — приклад відгуку з передбачуваною ймовірністю
def show_review_example(word):
    rows = products[products[word] > 0]

    if len(rows) == 0:
        print(f'Для слова "{word}" не знайдено відгуків.')
        return

    row = rows.iloc[0]
    one_review_df = pd.DataFrame([row])

    one_matrix, _ = get_feature_matrix(one_review_df, important_words)
    probability = predict_probability(one_matrix, coefficients)[0]
    predicted_sentiment = +1 if probability > 0.5 else -1

    print(f'Слово: {word}')
    print(f'Ймовірність позитивного відгуку: {probability:.4f}')
    print(f'Передбачений клас: {predicted_sentiment}')
    print(f'Реальний клас: {row["sentiment"]}')
    print('Текст відгуку:')
    print(row['review'][:700])


positive_word = positive_words.iloc[0]['word']
negative_word = negative_words.iloc[0]['word']

print('Приклад для позитивного слова')
show_review_example(positive_word)

print('\nПриклад для негативного слова')
show_review_example(negative_word)

Приклад для позитивного слова
Слово: baby
Ймовірність позитивного відгуку: 0.6592
Передбачений клас: 1
Реальний клас: 1
Текст відгуку:
My daughter had her 1st baby over a year ago. She did receive and fill up a First Year Calendar. When her son was nearing his first birthday she was looking for a Second Year Calendar to record his milestones. Thanks to Amazon I was able to get this for her and she LOVES it. Tender sweet art work - helpful stickers - unique pages to fill. A nice keepsake. A wonderful gift for a one-year old!

Приклад для негативного слова
Слово: won
Ймовірність позитивного відгуку: 0.6644
Передбачений клас: 1
Реальний клас: 1
Текст відгуку:
My baby is now almost 11 months old and is probably too big to really bathe in this tub anymore; however I continue to use it because it's just so easy to pop it on top of the kitchen sink and fill it up. He sits up in it now and splashes and enjoys bath-time. We bought several other bath tubs and this one won out, hands down.


**Найбільш позитивні слова:** *перелічіть їх тут*  
**Найбільш негативні слова:** *перелічіть їх тут*  
**Спостереження:** *прокоментуйте по одному слову з кожного списку*